In [1]:
import pandas as pd

eventos = pd.read_csv("../data/eventos_completo.csv")
playbooks = pd.read_csv("../data/playbooks.csv")

print(eventos.head())
print(playbooks)

  indicador        data  actual  forecast  diferenca  surpresa_zscore  \
0   CPI_EUA  2023-03-14     0.4       0.4        0.0         0.000000   
1   CPI_EUA  2023-04-12     0.1       0.2       -0.1        -1.007506   
2   CPI_EUA  2023-05-10     0.4       0.4        0.0         0.000000   
3   CPI_EUA  2023-06-13     0.1       0.2       -0.1        -1.007506   
4   CPI_EUA  2023-07-12     0.2       0.3       -0.1        -1.007506   

   atencao_bruta       IAN       ICE  
0           23.5  0.211111  0.329527  
1           28.0  0.311111  0.305865  
2           22.5  0.188889  0.282809  
3           20.0  0.133333  0.253607  
4           23.0  0.200000  0.229993  
  indicador ativo_alvo direcao_se_surpresa_positiva  \
0   CPI_EUA        SPY                       vender   

  direcao_se_surpresa_negativa  
0                      comprar  


In [2]:
LIMIAR_SURPRESA = 1.0
LIMIAR_IAN = eventos["IAN"].quantile(0.75)

print(f"Novo limiar de IAN (75º percentil): {LIMIAR_IAN:.4f}")

eventos["opera"] = (eventos["surpresa_zscore"].abs() > LIMIAR_SURPRESA) & (eventos["IAN"] > LIMIAR_IAN)

print(f"Total de eventos: {len(eventos)}")
print(f"Eventos operados: {eventos['opera'].sum()}")

Novo limiar de IAN (75º percentil): 0.2722
Total de eventos: 39
Eventos operados: 5


In [3]:
def determinar_direcao(row, playbooks):
    if not row["opera"]:
        return None
    regra = playbooks[playbooks["indicador"] == row["indicador"]].iloc[0]
    if row["surpresa_zscore"] > 0:
        return regra["direcao_se_surpresa_positiva"]
    else:
        return regra["direcao_se_surpresa_negativa"]

eventos["direcao"] = eventos.apply(lambda row: determinar_direcao(row, playbooks), axis=1)

print(eventos[["data", "surpresa_zscore", "IAN", "opera", "direcao"]])

          data  surpresa_zscore       IAN  opera  direcao
0   2023-03-14         0.000000  0.211111  False      NaN
1   2023-04-12        -1.007506  0.311111   True  comprar
2   2023-05-10         0.000000  0.188889  False      NaN
3   2023-06-13        -1.007506  0.133333  False      NaN
4   2023-07-12        -1.007506  0.200000  False      NaN
5   2023-08-10         0.000000  0.211111  False      NaN
6   2023-09-13         0.000000  0.122222  False      NaN
7   2023-10-12         1.007506  0.088889  False      NaN
8   2023-11-14        -1.007506  0.144444  False      NaN
9   2023-12-12         1.007506  0.044444  False      NaN
10  2024-01-11         1.007506  0.066667  False      NaN
11  2024-02-13         1.007506  0.255556  False      NaN
12  2024-03-12         0.000000  0.155556  False      NaN
13  2024-04-10         1.007506  0.311111   True   vender
14  2024-05-15        -1.007506  0.244444  False      NaN
15  2024-06-12        -1.007506  0.111111  False      NaN
16  2024-07-11

In [4]:
eventos["tamanho_posicao"] = eventos["IAN"] * (1 + eventos["ICE"])
eventos.loc[~eventos["opera"], "tamanho_posicao"] = 0

print(eventos[["data", "IAN", "ICE", "opera", "direcao", "tamanho_posicao"]])

          data       IAN       ICE  opera  direcao  tamanho_posicao
0   2023-03-14  0.211111  0.329527  False      NaN         0.000000
1   2023-04-12  0.311111  0.305865   True  comprar         0.406269
2   2023-05-10  0.188889  0.282809  False      NaN         0.000000
3   2023-06-13  0.133333  0.253607  False      NaN         0.000000
4   2023-07-12  0.200000  0.229993  False      NaN         0.000000
5   2023-08-10  0.211111  0.206212  False      NaN         0.000000
6   2023-09-13  0.122222  0.169503  False      NaN         0.000000
7   2023-10-12  0.088889  0.152624  False      NaN         0.000000
8   2023-11-14  0.144444  0.123645  False      NaN         0.000000
9   2023-12-12  0.044444  0.102096  False      NaN         0.000000
10  2024-01-11  0.066667  0.454807  False      NaN         0.000000
11  2024-02-13  0.255556  0.933170  False      NaN         0.000000
12  2024-03-12  0.155556 -0.568122  False      NaN         0.000000
13  2024-04-10  0.311111 -0.129396   True   vend

In [5]:
operacoes = eventos[eventos["opera"]]
print(operacoes[["data", "indicador", "surpresa_zscore", "IAN", "ICE", "direcao", "tamanho_posicao"]])
print(f"\nTotal de operações: {len(operacoes)}")

          data indicador  surpresa_zscore       IAN       ICE  direcao  \
1   2023-04-12   CPI_EUA        -1.007506  0.311111  0.305865  comprar   
13  2024-04-10   CPI_EUA         1.007506  0.311111 -0.129396   vender   
33  2026-02-13   CPI_EUA        -1.007506  0.433333 -0.127262  comprar   
35  2026-04-10   CPI_EUA        -1.007506  0.333333 -0.121264  comprar   
38  2026-07-14   CPI_EUA        -3.022518  0.322222 -0.106318  comprar   

    tamanho_posicao  
1          0.406269  
13         0.270855  
33         0.378187  
35         0.292912  
38         0.287964  

Total de operações: 5


In [6]:
eventos.to_csv("../data/eventos_com_decisao.csv", index=False)

In [1]:
import pandas as pd

eventos = pd.read_csv("../data/eventos_todos_completo.csv")
playbooks = pd.read_csv("../data/playbooks.csv")

LIMIAR_SURPRESA = 1.0

eventos["limiar_ian_indicador"] = eventos.groupby("indicador")["IAN"].transform(lambda x: x.quantile(0.75))

eventos["opera"] = (eventos["surpresa_zscore"].abs() > LIMIAR_SURPRESA) & (eventos["IAN"] > eventos["limiar_ian_indicador"])

print(eventos.groupby("indicador")["opera"].sum())

indicador
CPI_EUA    5
IPCA_BR    4
Name: opera, dtype: int64


In [2]:
def determinar_direcao(row, playbooks):
    if not row["opera"]:
        return None
    regra = playbooks[playbooks["indicador"] == row["indicador"]].iloc[0]
    if row["surpresa_zscore"] > 0:
        return regra["direcao_se_surpresa_positiva"]
    else:
        return regra["direcao_se_surpresa_negativa"]

eventos["direcao"] = eventos.apply(lambda row: determinar_direcao(row, playbooks), axis=1)

eventos["tamanho_posicao"] = eventos["IAN"] * (1 + eventos["ICE"])
eventos.loc[~eventos["opera"], "tamanho_posicao"] = 0

operacoes = eventos[eventos["opera"]]
print(operacoes[["data", "indicador", "surpresa_zscore", "IAN", "direcao", "tamanho_posicao"]])

          data indicador  surpresa_zscore       IAN  direcao  tamanho_posicao
1   2023-04-12   CPI_EUA        -1.007506  0.311111  comprar         0.406269
13  2024-04-10   CPI_EUA         1.007506  0.311111   vender         0.270855
33  2026-02-13   CPI_EUA        -1.007506  0.433333  comprar         0.378187
35  2026-04-10   CPI_EUA        -1.007506  0.333333  comprar         0.292912
38  2026-07-14   CPI_EUA        -3.022518  0.322222  comprar         0.287964
52  2024-02-01   IPCA_BR         1.570191  0.509434   vender         0.532847
63  2025-01-01   IPCA_BR        -1.237121  0.867925  comprar         1.206988
64  2025-02-01   IPCA_BR         4.182419  0.962264   vender         1.539319
65  2025-03-01   IPCA_BR         1.141957  0.641509   vender         0.570784


In [3]:
eventos.to_csv("../data/eventos_com_decisao.csv", index=False)